# Unified Poisson Solver Testing
This notebook runs the unified test suites for Problem 1 (Table 1 & 2) as well as the complete NUDFT/NUFFT Accuracy Comparisons, tracking strictly $L_2$ relative errors and computational runtimes.

In [1]:
import os, sys
import pandas as pd
# Main project root
repo_root = r"C:\Users\charl\OneDrive\Documents\GitHub\NUFFTRR_Poisson"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from Tests.CPU.testing_helpers import (
    run_tests_pipeline,
    render_accuracy,
    render_runtime,
    render_table2_accuracy,
    render_table2_runtime
)


# Setup

In [ ]:
fixed_N = 32
fixed_M = [32, 64, 128, 256]

MUTE_OUTPUT = True


# Run Problems

In [3]:
METHODS_P0 = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Fixed-jittered-NUDFT", label="Fixed jittered / NUDFT", azu_unif=1, mesh_kind="jittered", solver_azu_unif=1, use_nudft=True),
    dict(name="Fixed-jittered-NUFFT", label="Fixed jittered / NUFFT", azu_unif=1, mesh_kind="jittered", solver_azu_unif=1, use_nudft=False),
]

METHODS_P1 = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Fixed-rand-NUDFT", label="Fixed rand / NUDFT", azu_unif=1, mesh_kind="rand", solver_azu_unif=1, use_nudft=True),
    dict(name="Fixed-rand-NUFFT", label="Fixed rand / NUFFT", azu_unif=1, mesh_kind="rand", solver_azu_unif=1, use_nudft=False),
]

METHODS_COMP = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Unif-NUDFT", label="Uniform / NUDFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=1, use_nudft=True),
    dict(name="Unif-NUFFT", label="Uniform / NUFFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=1, use_nudft=False),
]
for kind in ("rand", "jittered", "clustered", "sine"):
    METHODS_COMP += [
        dict(name=f"Fixed-{kind}-NUDFT", label=f"Fixed {kind} / NUDFT", azu_unif=1, mesh_kind=kind, solver_azu_unif=1, use_nudft=True),
        dict(name=f"Fixed-{kind}-NUFFT", label=f"Fixed {kind} / NUFFT", azu_unif=1, mesh_kind=kind, solver_azu_unif=1, use_nudft=False),
    ]

print("Running Problem 0 - Table 1 (Jittered)...")
df_p0_t1 = run_tests_pipeline(N_vals_p0, M_vals_p0, fixed_other=None, methods=METHODS_P0, test_type="P1_Table1",mute=MUTE_OUTPUT)
print("\nRunning Problem 0 - Table 2 (Jittered)...")
df_p0_t2 = run_tests_pipeline(None, M_vals_p0, fixed_other=N_fixed_p0, methods=METHODS_P0, test_type="P1_Table2",mute=MUTE_OUTPUT)

print("Running Problem 1 - Table 1 (Rand)...")
df_t1 = run_tests_pipeline(N_vals_p1, M_vals_p1, fixed_other=None, methods=METHODS_P1, test_type="P1_Table1",mute=MUTE_OUTPUT)
print("\nRunning Problem 1 - Table 2 (Rand)...")
df_t2 = run_tests_pipeline(None, M_vals_p1, fixed_other=N_fixed_p1, methods=METHODS_P1, test_type="P1_Table2",mute=MUTE_OUTPUT)


print("Running Accuracy Comparisons...")
df_cN = run_tests_pipeline(N_vals_c, None, fixed_other=32, methods=METHODS_COMP, test_type="Accuracy_VaryN", mute=MUTE_OUTPUT)
df_cM = run_tests_pipeline(None, M_vals_c, fixed_other=32, methods=METHODS_COMP, test_type="Accuracy_VaryM", mute=MUTE_OUTPUT)
print("Done.")

Running Problem 0 - Table 1 (Jittered)...

Running Problem 0 - Table 2 (Jittered)...
Running Problem 1 - Table 1 (Rand)...

Running Problem 1 - Table 2 (Rand)...
Running Accuracy Comparisons...
Done.


# Accuracy

In [ ]:
print("--- ACCURACY SECTION ---")

print("Problem 0 - Table 2 - errors should decrease along rows")
render_table2_accuracy(df_p0_t2, title_prefix=f"Problem 0 - Table 2 (N={N_fixed_p0})")
print("Neumann error for nudft and nufft being constant is an issue. this is either with zero mode or handling fourier coeffs.")


print("Problem 1 - Table 2 - errors should decrease along rows")
render_table2_accuracy(df_t2, title_prefix=f"Problem 1 - Table 2 (N={N_fixed_p1})")
print("for nonuniform case dirichlet and neumann errors being constant is concerning. Indicates either mode issue or accuracy issues with algorithm")

print("Problem 2 - Table 2 -All errors should decrease along rows, and relatively same along columns")
render_accuracy(df_cM, index_col="M", columns_col="label", title_prefix="Accuracy Comparison (Fixed N=32)")
print("errors for diff meshes getting worse suggests nudft or nufft not accurate")

--- ACCURACY SECTION ---
Problem 0 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same

Problem 0 - Table 1 Accuracy


error along rows indicate nudft or nufft is not currently accurate enough
Problem 0 - Table 2 - errors should decrease along rows

Problem 0 - Table 2 (N=32) Accuracy


Neumann error for nudft and nufft being constant is an issue. this is either with zero mode or handling fourier coeffs.
Problem 1 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same

Problem 1 - Table 1 Accuracy


error along rows indicate nudft or nufft is not currently accurate enough. Inaccuracy is worse than Problem 0, likely due to random mesh being more challenging than jittered.
Problem 1 - Table 2 - errors should decrease along rows

Problem 1 - Table 2 (N=32) Accuracy


for nonuniform case dirichlet and neumann errors being constant is concerning. Indicates either mode issue or accuracy issues with algorithm
Problem 2 - Table 1 -All errors should be same along rows, and relatively same along columns

Accuracy Comparison (Fixed M=32) Accuracy


label,Uniform / FFT,Uniform / NUFFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
N,,,,,,
32,1.10e-05,1.10e-05,4.22e-04,1.10e-05,1.30e-01,1.07e-03
64,1.10e-05,1.10e-05,1.45e-03,1.10e-05,1.77e-01,1.40e-03
128,1.10e-05,1.10e-05,1.90e-03,1.10e-05,2.74e-01,2.79e-04
256,1.10e-05,1.10e-05,1.60e-03,1.10e-05,2.48e-01,1.36e-05
512,1.10e-05,1.10e-05,1.21e-03,1.10e-05,2.45e-01,1.10e-05


errors for diff meshes getting worse suggests nudft or nufft not accurate
Problem 2 - Table 2 -All errors should decrease along rows, and relatively same along columns

Accuracy Comparison (Fixed N=32) Accuracy


label,Uniform / FFT,Uniform / NUFFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
M,,,,,,
32,1.10e-05,1.10e-05,4.22e-04,1.10e-05,1.30e-01,1.07e-03
64,2.66e-06,2.66e-06,4.23e-04,2.66e-06,1.30e-01,1.07e-03
128,6.55e-07,6.55e-07,4.23e-04,6.55e-07,1.30e-01,1.07e-03
256,1.62e-07,1.62e-07,4.23e-04,1.62e-07,1.30e-01,1.07e-03
512,4.04e-08,4.04e-08,4.23e-04,4.05e-08,1.30e-01,1.07e-03


errors for diff meshes getting worse suggests nudft or nufft not accurate


# Runtimes

In [5]:
print("--- RUNTIME SECTION ---")
render_runtime(df_p0_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 0 (Jittered) Table 1")
render_table2_runtime(df_p0_t2, title_prefix=f"Problem 0 (Jittered) Table 2 (N={N_fixed_p0})")

render_runtime(df_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 1 (Rand) Table 1")
render_table2_runtime(df_t2, title_prefix=f"Problem 1 (Rand) Table 2 (N={N_fixed_p1})")

render_runtime(df_cN, index_col="N", columns_col="label", title_prefix="Accuracy Comparison (Fixed M=32)")
render_runtime(df_cM, index_col="M", columns_col="label", title_prefix="Accuracy Comparison (Fixed N=32)")

--- RUNTIME SECTION ---

Problem 0 (Jittered) Table 1 Runtime (s)



Problem 0 (Jittered) Table 2 (N=32) Runtime (s)



Problem 1 (Rand) Table 1 Runtime (s)



Problem 1 (Rand) Table 2 (N=32) Runtime (s)



Accuracy Comparison (Fixed M=32) Runtime (s)


label,Uniform / FFT,Uniform / NUFFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
N,,,,,,
32,8.81e-04,0.0288,0.4343,0.1985,0.8685,0.3099
64,9.23e-04,0.0280,0.5237,0.3707,1.0077,0.3835
128,0.0027,0.0438,0.5511,0.3887,0.8244,0.5197
256,0.0018,0.0461,0.5761,0.4662,0.6736,0.6017
512,0.0028,0.0512,0.6498,0.5317,0.7077,0.5034



Accuracy Comparison (Fixed N=32) Runtime (s)


label,Uniform / FFT,Uniform / NUFFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
M,,,,,,
32,9.52e-04,0.0271,0.6327,0.3050,0.7504,0.3852
64,0.0012,0.0350,0.8458,0.4358,1.1014,0.6064
128,0.0019,0.0474,1.4220,0.7430,1.7745,0.8312
256,0.0036,0.0727,2.5659,1.4262,2.9678,1.6547
512,0.0078,0.1464,4.7805,2.4438,6.5631,3.6405
